# Copyright (c) 2026 Khush Patil. All Rights Reserved.
# This code is for personal/educational viewing only. 
# Unauthorized copying, modification, or distribution is strictly prohibited.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import timm
import os

# --- 1. Setup Device & Transforms ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

import os

# --- 2. Auto-Detect and Load Datasets ---
train_dir = None
test_dir = None

# Scan Colab storage to find exactly where the folders unzipped
for root, dirs, files in os.walk('/content'):
    for d in dirs:
        if d.lower() == 'train':
            train_dir = os.path.join(root, d)
        elif d.lower() == 'test':
            test_dir = os.path.join(root, d)

if not train_dir or not test_dir:
    raise FileNotFoundError("❌ Could not find Train or Test folders. Double check the unzip command!")

print(f"✅ Auto-Detected Train Path: {train_dir}")
print(f"✅ Auto-Detected Test Path: {test_dir}")
print("-" * 50)

# Load the data using the newly found paths
train_data = datasets.ImageFolder(root=train_dir, transform=transform)
test_data = datasets.ImageFolder(root=test_dir, transform=transform)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=64, shuffle=False)

num_classes = len(train_data.classes)

# CRITICAL: Print this list and copy it for your app.py later!
print("COPY THIS LIST FOR YOUR APP.PY:")
print(train_data.classes)
print("-" * 50)

# --- 3. Define the Models ---
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

vit_model = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=num_classes)

class HybridModel(nn.Module):
    def __init__(self):
        super(HybridModel, self).__init__()
        resnet = timm.create_model('resnet18', pretrained=True)
        self.cnn_backbone = nn.Sequential(*list(resnet.children())[:-2])
        self.transformer_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.transformer_layer, num_layers=2)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn_backbone(x)
        x = x.flatten(2).permute(0, 2, 1)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.fc(x)

# --- 4. Training & Evaluation Functions ---
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def train_model(model, name, epochs=3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print(f"--- Training {name} ---")
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # Evaluate on test set after each epoch
        test_acc = evaluate_model(model, test_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Test Accuracy: {test_acc:.2f}%")

    # Save the model
    os.makedirs('models', exist_ok=True)
    save_path = f'models/{name}.pth'
    torch.save(model.state_dict(), save_path)
    print(f"Saved {name} to {save_path}\n")

# --- 5. Execute ---
cnn_model = SimpleCNN()
hybrid_model = HybridModel()
train_model(cnn_model, "cnn_model", epochs=5)
train_model(vit_model, "vit_model", epochs=3)
train_model(hybrid_model, "hybrid_model", epochs=4)

✅ Auto-Detected Train Path: /content/DevanagariHandwrittenCharacterDataset/Train
✅ Auto-Detected Test Path: /content/DevanagariHandwrittenCharacterDataset/Test
--------------------------------------------------
COPY THIS LIST FOR YOUR APP.PY:
['character_10_yna', 'character_11_taamatar', 'character_12_thaa', 'character_13_daa', 'character_14_dhaa', 'character_15_adna', 'character_16_tabala', 'character_17_tha', 'character_18_da', 'character_19_dha', 'character_1_ka', 'character_20_na', 'character_21_pa', 'character_22_pha', 'character_23_ba', 'character_24_bha', 'character_25_ma', 'character_26_yaw', 'character_27_ra', 'character_28_la', 'character_29_waw', 'character_2_kha', 'character_30_motosaw', 'character_31_petchiryakha', 'character_32_patalosaw', 'character_33_ha', 'character_34_chhya', 'character_35_tra', 'character_36_gya', 'character_3_ga', 'character_4_gha', 'character_5_kna', 'character_6_cha', 'character_7_chha', 'character_8_ja', 'character_9_jha', 'digit_0', 'digit_1', '

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

--- Training cnn_model ---
Epoch 1/5 | Loss: 0.8183 | Test Accuracy: 90.35%
Epoch 2/5 | Loss: 0.2188 | Test Accuracy: 91.87%
Epoch 3/5 | Loss: 0.1271 | Test Accuracy: 91.78%
Epoch 4/5 | Loss: 0.0833 | Test Accuracy: 91.28%
Epoch 5/5 | Loss: 0.0611 | Test Accuracy: 92.20%
Saved cnn_model to models/cnn_model.pth

--- Training vit_model ---
Epoch 1/3 | Loss: 2.7067 | Test Accuracy: 71.58%
Epoch 2/3 | Loss: 0.4196 | Test Accuracy: 90.57%
Epoch 3/3 | Loss: 0.1985 | Test Accuracy: 95.59%
Saved vit_model to models/vit_model.pth

--- Training hybrid_model ---
Epoch 1/4 | Loss: 0.2354 | Test Accuracy: 97.83%
Epoch 2/4 | Loss: 0.0934 | Test Accuracy: 98.57%
Epoch 3/4 | Loss: 0.0846 | Test Accuracy: 97.44%
Epoch 4/4 | Loss: 0.0786 | Test Accuracy: 98.55%
Saved hybrid_model to models/hybrid_model.pth

